# links.csv
Este notebook documenta el análisis exploratorio y limpieza del dataset **links.csv** de MovieLens.

**Entrada**: links.csv \
**Objetivos**: lectura, validación, limpieza y transformación \
**Salida**: links_clean.parquet

El dataset links.csv se ha utilizado para enriquecer la información de películas con llamadas a la API de TMDB. Durante este proceso, se han obtenido errores de dos tipos, lo cuales se han recogidos en requestTMD.logs: por un lado, se han encontrado películas con identificador de TMDB nulo y, por otro,  películas con identificador inexistente en la base de datos de TMDB. Estos casos se tratarán en el apartado de limpieza. La columna de identificador de IMDB no se ha empleado en el proyecto.

## Descripción del proceso

**Análisis y comprensión**
- movieId: identificador entero y único. CLAVE
- imdbId: identificador entero y único. 
- tmdbId: identificador entero y único. 

**Validación**
- Tipos de datos
- Valores nulos
- movieId único
- TmdbId único

**Limpieza**
- Para los valores nulos e inexistentes en la base de datos de TMDB, se procederá a eliminar las filas afectadas.

**Transformación**
- se elimina la columna imdbId
- Salida: archivo links.parquet


In [31]:
import pandas as pd

**Análisis y comprensión**

In [32]:
#lectura de datos
links = pd.read_csv("../data/01_raw/movielens/links.csv")
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [33]:
links.info()
#valores nulos en tmdbId
# la columna tmdbId se lee como float

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB


In [34]:
links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce').astype('Int64') # se usa Int64 ya que hay NaN

In [35]:
import os
os.path.exists("../logs/requestTMDB.log")

True

Se cargan los datos del archivo de errores tmdb.log

In [36]:
logs = pd.read_csv("../logs/requestTMDB.log", header = None, sep="|", names = ['timestamp','levelname','tmdbId', 'message'])
logs.head()

,timestamp,levelname,tmdbId,message
0,"2026-06-06 18:42:39,035",WARNING,tmdbId = 876,HTTP_Status = 404
1,"2026-06-06 18:44:47,553",WARNING,tmdbId = 2670,HTTP_Status = 404
2,"2026-06-06 18:55:27,134",WARNING,tmdbId = 7096,HTTP_Status = 404
3,"2026-06-06 18:56:18,174",WARNING,tmdbId = 8677,HTTP_Status = 404
4,"2026-06-06 18:58:14,545",WARNING,tmdbId = 9795,HTTP_Status = 404


In [ ]:
# se extraen  lod id de la columna tmdbId
tmdbId_logs = logs['tmdbId'].str.extract("([0-9]+)").astype('Int64')
# tmdbId_logs.columns = ['tmdbId']
# tmdbId_logs
# se eliminarán en el fichero de integridad referencial

## Validación

In [38]:
#  movieId sea entero
(links['movieId'] % 1 == 0).sum() == links['movieId'].size

np.True_

In [39]:
#se comprueba que movieId sea único
links['movieId'].nunique() == links['movieId'].size

True

In [40]:
#se comprueba que tmdbId sea único
links['tmdbId'].dropna().nunique()  == links['tmdbId'].dropna().size

False

In [41]:
duplicados = links[links['tmdbId'].duplicated(keep=False)]
duplicados

,movieId,imdbId,tmdbId
624,791,113610,<NA>
843,1107,102336,<NA>
2141,2851,81454,<NA>
3027,4051,56600,<NA>
4169,6003,290538,4912
5532,26587,92337,<NA>
5854,32600,377059,<NA>
6059,40697,105946,<NA>
7382,79299,874957,<NA>
9106,144606,270288,4912


In [42]:
# si hubiese duplicados se deberían eliminar, por tanto se conservaría solo uno
# links = links.drop_duplicates(subset="movieId", keep="first")

## Limpieza

In [43]:
#Se eliminan los valores nulos
links = links.dropna()
links.info()

<class 'pandas.DataFrame'>
Index: 9734 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9734 non-null   int64
 1   imdbId   9734 non-null   int64
 2   tmdbId   9734 non-null   Int64
dtypes: Int64(1), int64(2)
memory usage: 313.7 KB


## Transformación

In [45]:
# se elimina la columna imdbId
links = links[['movieId','tmdbId']]

links.to_parquet('../data/02_processed/links_clean.parquet', index=False)